In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from lp_analysis import load_and_preprocess

pd.options.plotting.backend = "plotly"
pd.options.display.max_columns=100
pd.options.display.max_rows = 50


stem = 'weth_usdc'
df = load_and_preprocess("data/weth_usdc_500.csv", n_rows=None)

high_fee = pd.read_parquet(f'data/lp_analysis_output_{stem}_3000.parquet')
low_fee = pd.read_parquet(f"data/lp_analysis_output_{stem}_500.parquet")

In [3]:
low_fee['fee_tier'] = 'low_fee'
high_fee['fee_tier'] = 'high_fee'
for _df in [low_fee,high_fee]:
    _df["time_in_range/life_time"] = (
        _df["time_in_range"] / _df["lifetime_hours"]
    )


In [4]:
df = pd.concat([low_fee,high_fee],axis=0)

In [5]:
df["fee_pct"] = df.total_fee_CM2M / df.total_fee_CM2M.sum()
df["volume_pct"] = df.volume / df.volume.sum()

In [6]:
df.plot(
    x="volume_pct",
    y="fee_pct",
    kind="scatter",
    color="fee_tier",
    title="Fees & Volume, Tiers = 500,3000",
)

In [7]:
# Open question: with one pool, whether you use fees or volume, you get the same ownerhsip of rewards. with two, at the ind level, you might get different ownerships.
# pool 1: f1
# - A
# - B
# pool 2: f2
#

In [8]:
df.plot(x='volume',y='total_fee_CM2M',kind='scatter',color='fee_tier',title='Fees & Volume, Tiers = 500,3000')

In [9]:

df.drop(columns=['liq_idx','liq_val']).groupby('fee_tier').sum().plot(y=['volume','avg_liq'],kind='bar',barmode='group',title='Volume & Liquidity')

In [10]:
df.drop(columns=["liq_idx", "liq_val",'lp_tuple']).groupby("fee_tier").mean().plot(y=["volume", "avg_liq"], kind="bar", barmode="group", title="Volume & Liquidity Per Position"
)

In [11]:
df.drop(columns=["liq_idx", "liq_val", "lp_tuple"]).groupby("fee_tier").mean().plot(
    y=["total_fee_CM2M"],
    kind="bar",
    barmode="group",
    title="Total Fee Value Per Position",
)

In [12]:
# lp_tuple: MANY people with MANY positions <-- groupby address.

In [13]:
df['roi_time_weighted/avg_liq'] = df['roi_time_weighted']/(df['avg_liq']/10**12)

In [77]:
dg = pd.concat([df[df.fee_tier == "high_fee"], df[df.fee_tier == "low_fee"]],axis=0)
dg.plot(
    x="price_apprec",
    y="roi_time_weighted",
    kind="scatter",
    color="fee_tier",
    title="",
)

In [78]:
dg.plot(
    x="price_apprec",
    y="fee_roi",
    kind="scatter",
    color="fee_tier",
    title="",
)

In [79]:
dg.plot(
    x="price_apprec",
    y="inv_roi",
    kind="scatter",
    color="fee_tier",
    title="",
    trendline='ols',
)


In [36]:

dg.plot(
    x="price_apprec",
    y="inv_roi",
    kind="scatter",
    color="fee_tier",
    title="",
)

In [37]:
dg.plot(
    x="price_apprec",
    y="il_time_weighted",
    kind="scatter",
    color="fee_tier",
    title="",
    opacity=1,
)

In [22]:
numeric_cols = df._get_numeric_data().columns 
c = df[numeric_cols].corr()
c['roi_time_weighted'].sort_values(ascending=False).plot()

In [23]:
c["fee_roi"].sort_values(ascending=False).plot()

In [24]:
dg.plot(
    x="lifetime_hours",
    y="fee_roi",
    kind="scatter",
    color="fee_tier",
    title="",
    opacity=1,
)

In [38]:
dg['lp_id'] = dg.lp_tuple.apply(lambda x: x.strip("()").split(',')[0])

In [23]:
from scipy.stats.mstats import winsorize

In [24]:
df["avg_liq*lifetime_hours_w"] = winsorize(df["avg_liq*lifetime_hours"],limits=[0.05,0.05])
df.plot(x="avg_liq*lifetime_hours_w",kind='hist',color='fee_tier',nbins=100)

In [ ]:
dg.plot(
    x="price_vol",
    y="_time_weighted",
    kind="scatter",
    color='fee_tier',
)

In [34]:
dg.plot(
    x="price_vol",
    y="il_time_weighted",
    kind="scatter",
    color="fee_tier",
)

In [35]:
dg.plot(
    x="price_apprec",
    y="il_time_weighted",
    kind="scatter",
    color="fee_tier",
)

In [57]:
dg["total_fee_CM2M/avg_liq"] = dg["total_fee_CM2M"] / dg["avg_liq"]

dg[dg['fee_tier']=='high_fee'].plot(
    y="roi_time_weighted",
    x="il_time_weighted",
    kind="scatter",
    color="time_in_range/life_time",
    opacity=0.6,
)

In [28]:

df['pool'] = np.where(df['fee_rate']==500,'low','high')

In [29]:
for _df in [high_fee,low_fee]:
    # quantities of interest at the event level
    # _df.ffill(inplace=True)
    _df["evt_block_time"] = pd.to_datetime(_df["evt_block_time"])
    # totals across lps
    _df['liquidity_at_tick'] = _df['pool_liquidity'].astype(float).ffill()/10**18
    _df["total_liquidity"] = _df.filter(regex="\\)$").sum(axis=1) / 10**18
    _df["liquidity_at_tick_to_total"] = (_df["liquidity_at_tick"] / _df["total_liquidity"])
    _df["total_fee_0"] = _df.filter(regex="_fee_0").sum(axis=1)/10**18
    _df["total_fee_1"] = _df.filter(regex="_fee_1").sum(axis=1) / 10**18
    _df["total_inv_0"] = _df.filter(regex="_inventory_0").sum(axis=1) / 10**18
    _df["total_inv_1"] = _df.filter(regex="_inventory_1").sum(axis=1) / 10**18
    # accumulations & values of things M2M
    _df["total_fee_0_acc"] = _df["total_fee_0"].cumsum()
    _df["total_fee_1_acc"] = _df["total_fee_1"].cumsum()
    _df["total_fee_acc_M2M"] = (
        _df["total_fee_0_acc"] + (1 / _df["price"]) * _df["total_fee_1_acc"]
    )
    # "tvl"
    _df["total_inv_value_M2M"] = (
        _df["total_inv_0"] + (1 / _df["price"]) * _df["total_inv_1"]
    )

    # swap stuff
    _df["swaps.dollars_in"] = np.where(
        _df["event"].eq("swap") & (_df["amount0"] > 0), _df["amount0"], 0
    ) + np.where(
        _df["event"].eq("swap") & (_df["amount1"] > 0),(1 / _df["price"])*_df["amount1"], 0
    )/10**18
    _df["swaps.dollars_in_acc"] = _df["swaps.dollars_in"].cumsum()/10**18

    _df["num_swaps"] = _df['event'].eq("swap").cumsum()
    _df["num_mints"] = _df["event"].eq("mint").cumsum()
    _df["num_burns"] = _df["event"].eq("burn").cumsum()
    # lp stuff
    _df["num_alive_lps"] = (_df.filter(regex="\\)$")>0).sum(axis=1)
    _df["total_liquidity_per_lp"] = _df['total_liquidity']/_df['num_alive_lps']
    _df["dollar_fees_acc_per_unit_liquidity"] = (
        _df["total_fee_acc_M2M"] / _df["total_liquidity"]
    )
    _df["dollar_fees_acc_per_lp"] = _df["total_fee_acc_M2M"]/_df["num_alive_lps"]

    _df["dollar_fees_per_lp"] = _df["total_fee_acc_M2M"] / _df["num_alive_lps"]

# 100000
# 45000 swap,mints,burns
# 145000 row thing --> 75 days

regex = "date|total|swaps|num|liquidity_at_tick|total_liquidity_per_lp|dollar_fees_acc_per_unit_liquidity|dollar_fees_acc_per_lp"

df = pd.merge(
        high_fee.filter(regex=regex).groupby('date').last().add_prefix("high_fee."),
        low_fee.filter(regex=regex).groupby('date').last().add_prefix("low_fee."),left_index=True, right_index=True, how="outer")

# add ratios at the HOUR level
for col in df.filter(regex='high_fee.').columns:
    quantity = col.lstrip("high_fee.")
    df[f'low2high.{quantity}'] = df[f'low_fee.{quantity}']/df[f'high_fee.{quantity}']
pd.options.plotting.backend='plotly'
df.plot(template='plotly_dark')

KeyError: 'evt_block_time'

In [ ]:
dg["lp_id"] = dg.lp_tuple.apply(lambda x: x.strip("()").split(",")[0])

In [45]:
dg.groupby('lp_id')['lp_tuple'].count().idxmax()

"'0xa57bd00134b2850b2a1c55860c9e9ea100fdd6cf'"

In [64]:
dg.groupby("lp_id")['lp_tuple'].count()

lp_id
'0x00062acefae5c760cc18cac5a98d84e0ced0cfd1'    1
'0x000ef2b60d565ac16d06ff791129055e591b631e'    1
'0x0012a7f00af8a643ba5a6aa187f915b4c13289df'    1
'0x0027a46efb18a6d72f113f8ebf4328ad65df22fa'    9
'0x00336cd9f823dd8b5c5741638e5038fd561f01b9'    3
                                               ..
'0xffe505d1753602e36cf91e97cf71ac2a328f2ae6'    1
'0xffe6212baf6c88850dcd6511cd32be11c50d3a61'    2
'0xffeb7a444cddcbf4c9add4f8df158bc4e5ea6445'    1
'0xffefdcfff613c9bbb9928f6ff44f07c7b562bfdf'    3
'0xfff0971d9cff7e47d894533fc0a297c6c7aaf348'    3
Name: lp_tuple, Length: 7089, dtype: int64

In [66]:
dg.groupby("lp_id")["lp_tuple"].count().quantile(1)

np.float64(412.0)

In [51]:
dg[dg.lp_id == "'0xa57bd00134b2850b2a1c55860c9e9ea100fdd6cf'"].plot(x='fee_tier',y='roi_time_weighted',kind='scatter')

In [75]:
dg.groupby(["lp_id",'fee_tier'])[['price_apprec','roi_time_weighted']].mean().reset_index().plot(x='price_apprec',y='roi_time_weighted',kind='scatter',color='fee_tier')

In [76]:
dg.groupby(["lp_id", "fee_tier"])[
    ["price_apprec", "fee_roi"]
].mean().reset_index().plot(
    x="price_apprec", y="fee_roi", kind="scatter", color="fee_tier"
)

In [ ]:
dg.groupby(["lp_id", "fee_tier"])[
    ["price_apprec", "roi_time_weighted"]
].mean().reset_index()

,lp_id,fee_tier,price_vol,roi_time_weighted
0,'0x00062acefae5c760cc18cac5a98d84e0ced0cfd1',high_fee,7.029921e+07,-0.203469
1,'0x000ef2b60d565ac16d06ff791129055e591b631e',low_fee,1.522359e+07,-0.057723
2,'0x0012a7f00af8a643ba5a6aa187f915b4c13289df',high_fee,7.916525e+06,-0.034835
3,'0x0027a46efb18a6d72f113f8ebf4328ad65df22fa',high_fee,3.343188e+07,0.014548
4,'0x0027a46efb18a6d72f113f8ebf4328ad65df22fa',low_fee,4.869658e+05,-0.000634
...,...,...,...,...
8004,'0xffe6212baf6c88850dcd6511cd32be11c50d3a61',low_fee,8.810225e+06,0.004080
8005,'0xffeb7a444cddcbf4c9add4f8df158bc4e5ea6445',low_fee,7.108368e+07,-0.245304
8006,'0xffefdcfff613c9bbb9928f6ff44f07c7b562bfdf',high_fee,3.138247e+07,-0.063312
8007,'0xffefdcfff613c9bbb9928f6ff44f07c7b562bfdf',low_fee,4.958674e+07,-0.167996
